# MAP-Elites Track Generations

In [14]:
import sys
import os

# Get the current working directory
cwd = os.getcwd()
print(f"Current Working Directory: {cwd}")

# Define the path to the 'mapelite' folder
# We assume the notebook is running from the root 'Quality-Diversity-...' folder
mapelite_path = os.path.join(cwd, 'mapelite')

# Add it to the system path so Python can find config.py, utils.py, etc.
if mapelite_path not in sys.path:
    sys.path.append(mapelite_path)
    print(f"Added '{mapelite_path}' to sys.path")

Current Working Directory: D:\dev\Quality-Diversity-for-Racing-Track-Design


In [15]:
import numpy as np
import requests
import random
import matplotlib.pyplot as plt
import joblib    
import glob
import os
import pickle

from ribs.archives import SlidingBoundariesArchive
from ribs.schedulers import Scheduler
from ribs.visualize import sliding_boundaries_archive_heatmap
from dask.distributed import Client, LocalCluster, as_completed

# --- Import Modular Code ---
from config import (
    ITERATIONS, BATCH_SIZE, INVALID_SCORE, CHECKPOINT_EVERY, 
    CHECKPOINT_DIR, HEATMAP_DIR, SOLUTION_DIM, ARCHIVE_BINS, 
    REMAPPING_EVERY, BUFFER_SIZE, INIT_POPULATION
)
from utils import evaluate_solution, array_to_solution, fitness_formula, get_initial_archive_ranges
from emitter import CustomEmitter




In [16]:
# --- Initialize directories ---
# Create directories if they don't exist
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(HEATMAP_DIR, exist_ok=True)

In [17]:
# --- DASK SETUP ---
print("Setting up Dask LocalCluster...")
cluster = LocalCluster(processes=True, n_workers=BATCH_SIZE, threads_per_worker=1)
client = Client(cluster)
print(f"Dask Dashboard link: {client.dashboard_link}")

Setting up Dask LocalCluster...


D:\dev\Quality-Diversity-for-Racing-Track-Design\venv\Lib\site-packages\distributed\node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 61414 instead
  warnings.warn(
C:\Users\milob\AppData\Local\Programs\Python\Python312\Lib\contextlib.py:144: UserWarning: Creating scratch directories is taking a surprisingly long time. (1.11s) This is often due to running workers on a network file system. Consider specifying a local-directory to point workers to write scratch data to a local disk.
  next(self.gen)


Dask Dashboard link: http://127.0.0.1:61414/status


In [18]:
ranges = get_initial_archive_ranges()

# --- Initialize Archive and Scheduler ---
archive = SlidingBoundariesArchive(
    solution_dim=SOLUTION_DIM,
    dims=[ARCHIVE_BINS, ARCHIVE_BINS], # UMAP dimensions
    ranges=ranges,         # UMAP descriptor ranges
    remap_frequency=REMAPPING_EVERY,
    buffer_capacity=BUFFER_SIZE
)

emitter = CustomEmitter(
    archive,
    solution_dim=SOLUTION_DIM,
    batch_size=BATCH_SIZE,
    # Last element (ID) can be arbitrarily large (ID=iteration + fractional_part)
    bounds=[(0, 600)] * (SOLUTION_DIM - 1) + [(0, float("inf"))]
)

scheduler = Scheduler(archive, [emitter])

# --------------------------------------------------------------
# Resume from latest pickle checkpoint if one exists
# --------------------------------------------------------------
checkpoints = sorted(glob.glob(f"{CHECKPOINT_DIR}checkpoint_*.pkl"))
start_iter = 0
global_best_score = INVALID_SCORE
global_best_id = None

if checkpoints:
    latest_ckpt = checkpoints[-1]
    with open(latest_ckpt, "rb") as f:
        state = pickle.load(f)
    scheduler           = state["scheduler"]
    # Get the latest archive instance from the scheduler
    archive             = scheduler.archive
    start_iter          = state["iteration"]
    global_best_score   = state["global_best_score"]
    global_best_id      = state["global_best_id"]
    print(f"[Resume] Loaded {latest_ckpt}, resuming from iteration {start_iter+1}")
else:
    print("[Resume] No checkpoint found — starting fresh.")


def run_map_elites(total_iters, start_iter=0):
    # declare that we mean to update these module-level variables
    global global_best_score, global_best_id
    
    for i in range(start_iter, total_iters):
        print(f"\n=== Starting iteration {i+1} ===")
        
        # 1. Ask the emitter for a batch of solutions
        sols = scheduler.ask()
        sol_dicts = [array_to_solution(s) for s in sols]
        
        # 2. Evaluate solutions in parallel using Dask
        futs = [client.submit(evaluate_solution, sol) for sol in sol_dicts]
        # Block until all evaluations are complete
        gathered = [f.result() for f in as_completed(futs)]

        obj_list, clean_solutions = [], []
        
        for sol_id, ok, msg, score, desc in gathered:
            
            if not ok:
                print(f"Warning: clamping bad score for ID={sol_id} ({msg})")
                score = INVALID_SCORE
            else:
                # Note: fitness_formula was already called inside evaluate_solution 
                # to calculate 'score' and update global _STATS.\n
                print(f"Solution ID={sol_id} evaluated with score={score:.2f}")
                if score > global_best_score:
                    global_best_score = score
                    global_best_id = sol_id
            
            clean_solutions.append((score, desc))
            obj_list.append(score)

        # 3. Tell the scheduler the results
        obj_batch, meas_batch = zip(*clean_solutions)
        scheduler.tell(list(obj_batch), list(meas_batch))

        # 4. Log statistics
        batch_best = max(obj_list) if obj_list else INVALID_SCORE
        print(f"Iteration {i+1} ended. Best in batch = {batch_best:.2f}")
        print(f"Global best so far: {global_best_score:.2f} (ID={global_best_id})")

        data_archive = archive.data()
        if data_archive:
            arch_obj = data_archive["objective"]
            valid = arch_obj != INVALID_SCORE
            mean_val = np.mean(arch_obj[valid]) if np.any(valid) else 0.0
            best_val = np.max(arch_obj[valid]) if np.any(valid) else 0.0
            cov = archive.stats.coverage
            print(f"Archive size={len(archive)}, cov={cov:.3f}, mean={mean_val:.2f}, best={best_val:.2f}")
        else:
            print("Archive still empty")

        # 5. Checkpoint
        if (i + 1) % CHECKPOINT_EVERY == 0:
            ckpt_name = f"{CHECKPOINT_DIR}checkpoint_{i+1:04d}.pkl"
            with open(ckpt_name, "wb") as f:
                pickle.dump({
                    "scheduler": scheduler,
                    "iteration": i+1,
                    "global_best_score": global_best_score,
                    "global_best_id": global_best_id
                }, f)
            print(f"[Checkpoint] Saved {ckpt_name}")

        # 6. Plot Heatmap
        if (i + 1) % 10 == 0:
            arch_obj = np.array(archive.data()["objective"])
            valid = arch_obj != INVALID_SCORE
            
            if np.any(valid):
                p_low, p_high = np.percentile(arch_obj[valid], [20, 80])
            else:
                p_low = p_high = 0.0

            fig = plt.figure(figsize=(8, 6))
            sliding_boundaries_archive_heatmap(
                archive,
                boundary_lw=0.5,
                vmin=p_low,
                vmax=p_high,
            )
            plt.title(f"Archive Heat-map – Iteration {i+1}")
            plt.xlabel("UMAP-1")
            plt.ylabel("UMAP-2")
            plt.tight_layout()
            plt.savefig(f"{HEATMAP_DIR}archive_heatmap_iter_{i+1}.png")
            plt.close(fig)

[Archive Init] Auto-detected ranges: X=(np.float32(0.5492356), np.float32(20.738216)), Y=(np.float32(-1.5278269), np.float32(13.583673))
[Resume] Loaded data/checkpoints\checkpoint_0400.pkl, resuming from iteration 401


In [19]:
# Run the main loop
run_map_elites(ITERATIONS, start_iter)


=== Starting iteration 401 ===
Emitter.ask() called for iteration 401
Crossover solutions for iteration 401
Solution ID=400.0398512013261 evaluated with score=0.00
Solution ID=400.0398512013261 evaluated with score=0.00
Solution ID=400.00060722649863 evaluated with score=0.00
Solution ID=400.00060722649863 evaluated with score=0.00
Solution ID=400.43789217133065 evaluated with score=0.00
Solution ID=400.43789217133065 evaluated with score=0.00
Solution ID=400.6033641401109 evaluated with score=0.00
Solution ID=400.6033641401109 evaluated with score=0.00
Solution ID=400.6651243120135 evaluated with score=0.00
Solution ID=400.6651243120135 evaluated with score=0.00
Iteration 401 ended. Best in batch = 0.00
Global best so far: 150.25 (ID=52.80978615386888)
Archive size=504, cov=0.560, mean=21.37, best=150.25

=== Starting iteration 402 ===
Emitter.ask() called for iteration 402
Mutating solutions for iteration 402
Solution ID=401.5333498890946 evaluated with score=0.00
Solution ID=401.24

KeyboardInterrupt: 

In [ ]:
data      = archive.data()
objs      = data["objective"]
size      = len(archive)
coverage  = archive.stats.coverage

print(f"Final archive size={size}, Coverage={coverage:.3f}")
print(f"Archive size     : {size}")
print(f"Coverage         : {coverage:.3%}")
print(f"Best objective   : {objs.max():.3f}")
print(f"Mean objective   : {objs[objs != INVALID_SCORE].mean():.3f}\n")

k = 10                                        # how many to list
for i in range(min(k, size)):
    sol_vec = data["solution"][i]
    sol_id  = sol_vec[-1]
    umap_xy = data["measures"][i]
    print(f"{i+1:2d}. "
          f"ID={sol_id:.17f}   "
          f"UMAP=({umap_xy[0]:7.3f}, {umap_xy[1]:7.3f})")

In [ ]:
from utilities import array_to_solution # Re-importing array_to_solution here just in case, though it's technically in the imports already.
from utilities import _STATS # Import global stats from utilities for inspection

target_x = 10.0   # change this as needed
k = 4

measures = data["measures"]                    # (N, 2) array
dx       = np.abs(measures[:, 0] - target_x)   # distance to target on x-axis

nearest_idx = np.argsort(dx)[:k]               # indices of the k closest

for rank, idx in enumerate(nearest_idx, 1):
    sol_vec = data["solution"][idx]
    sol_id  = sol_vec[-1]
    umap_xy = measures[idx]
    score   = objs[idx]
    print(f"{rank:2d}. "
          f"ID={sol_id:.17f}   "
          f"dx={dx[idx]:7.3f}   "
          f"score={score:8.3f}   "
          f"UMAP=({umap_xy[0]:7.3f}, {umap_xy[1]:7.3f})")


# ----------------------------------------------------------
# Plotting the scatter + highlight
# ----------------------------------------------------------

all_x, all_y = measures[:, 0], measures[:, 1]

plt.figure(figsize=(8, 6))
sc = plt.scatter(all_x, all_y,
                 c=objs,                # colour scale = performance
                 s=12,                  # dot size
                 alpha=0.8,
                 cmap="viridis")

# highlight the nearest k in red with black edge
highlight = measures[nearest_idx]
plt.scatter(highlight[:, 0], highlight[:, 1],
            facecolors='none',
            edgecolors='red',
            s=80,
            linewidths=1.5,
            label=f"{len(nearest_idx)} closest to UMAP-x={target_x}")

plt.colorbar(sc, label="Objective value")
plt.title("Archive in UMAP space\n(highlighted = closest to target x)")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
m = 15                                   # how many random elites to inspect
rng = np.random.default_rng(42)          # reproducible
rand_idx = rng.choice(len(archive), size=m, replace=False)

samples = []                             # keep the decoded solutions here

for j, idx in enumerate(rand_idx, 1):
    sol_vec = data["solution"][idx]
    sol_id  = sol_vec[-1]
    umap_xy = data["measures"][idx]
    score   = objs[idx]
    print(f"{j:2d}. ID={sol_id:.17f}   "
          f"score={score:8.3f}   "
          f"UMAP=({umap_xy[0]:7.3f}, {umap_xy[1]:7.3f})")
    
    # store full dict in `samples` for later, if desired
    samples.append(array_to_solution(sol_vec))


# --------------------------------------------------------------
# Plotting the 2D Histogram
# --------------------------------------------------------------

# Filter INVALID_SCOREs before computing mean/std
valid_objs = objs[objs != INVALID_SCORE]
if len(valid_objs) > 0:
    obj_mean = valid_objs.mean()
    obj_std  = valid_objs.std()
else:
    obj_mean = 0.0
    obj_std = 0.0

# Set outlier bounds based on the mean of valid objectives
lo, hi   = obj_mean - 3*obj_std, obj_mean + 3*obj_std

# Create mask by filtering out INVALID_SCOREs and outliers
valid_mask = objs != INVALID_SCORE
outlier_mask = (objs[valid_mask] >= lo) & (objs[valid_mask] <= hi)
final_mask = np.zeros_like(objs, dtype=bool)
final_mask[valid_mask] = outlier_mask

x_f, y_f = all_x[final_mask], all_y[final_mask]
w_f      = objs[final_mask]                       # colour = objective value

plt.figure(figsize=(8, 6))
# Only plot if there are valid and non-outlier points remaining
if len(x_f) > 0:
    hb = plt.hist2d(x_f, y_f,
                    bins=50,
                    weights=w_f,
                    cmap="plasma",
                    # Centering colour bar around the mean 
                    vmin=obj_mean - obj_std,
                    vmax=obj_mean + obj_std)

    plt.colorbar(hb[3], label="Mean objective in bin")
else:
    plt.text(0.5, 0.5, "No valid data to plot after outlier filtering.", 
             ha='center', va='center', transform=plt.gca().transAxes)
    
plt.title("Performance landscape (outliers clipped, scale ~ mean ±1 σ)")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.tight_layout()
plt.show()